# 01 — Extraction des données brutes

Ce notebook télécharge le dataset **RLCS 2021-22** depuis Kaggle et le place dans `data/raw/`.

**Source :** [RLCS 2021-22 — Kaggle](https://www.kaggle.com/datasets/dylanmonfret/rlcs-202122)

**Pipeline ETL :**
```
01_extract (ce notebook)    Kaggle → data/raw/*.csv
02_transform                data/raw/*.csv → data/processed/ (nettoyage, normalisation)
03_load                     data/processed/ → DuckDB via SQLAlchemy + Alembic
```

## 1. Configuration

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
KAGGLE_DATASET = 'dylanmonfret/rlcs-202122'

print(f'Racine du projet : {ROOT}')
print(f'Dossier cible    : {RAW_DIR}')

## 2. Téléchargement depuis Kaggle

Utilise `kagglehub` pour télécharger le dataset.
Si les CSV sont déjà présents dans `data/raw/`, l'étape est ignorée.

In [ ]:
import shutil
import kagglehub

existing_csvs = list(RAW_DIR.glob('*.csv'))

if len(existing_csvs) >= 6:
    print(f'CSV d\u00e9j\u00e0 pr\u00e9sents ({len(existing_csvs)} fichiers) \u2014 t\u00e9l\u00e9chargement ignor\u00e9.')
else:
    print(f'T\u00e9l\u00e9chargement du dataset {KAGGLE_DATASET}...')
    download_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print(f'T\u00e9l\u00e9charg\u00e9 dans : {download_path}')

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for csv_file in sorted(download_path.rglob('*.csv')):
        dest = RAW_DIR / csv_file.name
        shutil.copy2(csv_file, dest)
        print(f'  \u2192 {csv_file.name}')

    print(f'Fichiers copi\u00e9s dans {RAW_DIR}')

## 3. Inventaire des fichiers

In [ ]:
csv_files = sorted(RAW_DIR.glob('*.csv'))

rows = []
for f in csv_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    with open(f, 'r', encoding='utf-8') as fh:
        header = fh.readline().strip().split(',')
        n_lines = sum(1 for _ in fh)
    rows.append({
        'fichier': f.name,
        'lignes': n_lines,
        'colonnes': len(header),
        'taille_mb': round(size_mb, 2)
    })

df_inv = pd.DataFrame(rows)
display(df_inv)
print(f'Total : {df_inv["lignes"].sum():,} lignes | {df_inv["taille_mb"].sum():.1f} MB')

## 4. Aperçu de chaque fichier

3 premières lignes + liste des colonnes par catégorie.

In [ ]:
for f in csv_files:
    df = pd.read_csv(f, nrows=3)
    print(f'\n{"=" * 60}')
    print(f'{f.name}  ({len(df.columns)} colonnes, {rows[[r["fichier"] for r in rows].index(f.name)]["lignes"]:,} lignes)')
    print(f'{"=" * 60}')
    print(f'Colonnes : {list(df.columns[:12])}{" ..." if len(df.columns) > 12 else ""}')
    display(df)

## 5. Qualité des données

Vérification rapide : valeurs nulles, types, doublons sur les clés.

In [ ]:
# Cles primaires attendues par fichier
primary_keys = {
    'main.csv': ['game_id'],
    'players_db.csv': ['player_id'],
    'games_by_players.csv': ['game_id', 'player_id'],
    'games_by_teams.csv': ['game_id', 'team_id'],
    'matches_by_players.csv': ['match_id', 'player_id'],
    'matches_by_teams.csv': ['match_id', 'team_id'],
}

for f in csv_files:
    df = pd.read_csv(f)
    pk = primary_keys.get(f.name, [])

    null_pct = df.isnull().mean() * 100
    cols_with_nulls = null_pct[null_pct > 0].sort_values(ascending=False)

    print(f'\n--- {f.name} ---')

    # Doublons sur la cle primaire
    if pk:
        n_dupes = df.duplicated(subset=pk).sum()
        print(f'  PK {pk} : {n_dupes} doublons{" !!" if n_dupes > 0 else ""}')

    # Colonnes avec le plus de nulls
    if len(cols_with_nulls) > 0:
        print(f'  Colonnes avec nulls ({len(cols_with_nulls)}) :')
        for col, pct in cols_with_nulls.head(5).items():
            print(f'    {col:<45} {pct:>5.1f}%')
        if len(cols_with_nulls) > 5:
            print(f'    ... et {len(cols_with_nulls) - 5} autres')
    else:
        print(f'  Aucun null')

## 6. Résumé

Les données brutes sont extraites dans `data/raw/`. Prochaine étape : **02_transform** (nettoyage et normalisation).

In [ ]:
print('Extraction termin\u00e9e.')
print(f'  Fichiers : {len(csv_files)} CSV dans {RAW_DIR}')
print(f'  Volume   : {df_inv["lignes"].sum():,} lignes | {df_inv["taille_mb"].sum():.1f} MB')
print(f'\nProchaine \u00e9tape : 02_transform.ipynb')